<a href="https://colab.research.google.com/github/Simhaa-code/Autonomous-Bot-files-/blob/main/FPproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy scikit-learn plotly folium gradio

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import folium
from folium.plugins import HeatMap

import gradio as gr

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ==========================================
# LOAD DATASET
# ==========================================

CSV_URL = "https://raw.githubusercontent.com/dhanushkumar76/Smart-City-Data-Analytics/refs/heads/main/python2222222/updated/Banglore_traffic_Dataset.csv"

traffic = pd.read_csv(CSV_URL)

print("Dataset loaded successfully.")
print("Rows:", traffic.shape[0])
print("Columns:", traffic.shape[1])

display(traffic.head())

Dataset loaded successfully.
Rows: 8936
Columns: 16


,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,70.632330,84.044600,85.403629,111,Clear,No
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,41.924899,91.407038,59.983689,100,Clear,No
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,44.662384,61.375541,95.466020,189,Clear,No
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,32.773123,75.547092,63.567452,111,Clear,No
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,35.092601,64.634762,93.155171,104,Clear,No


In [ ]:
print("=" * 60)
print("ORIGINAL DATASET INFORMATION")
print("=" * 60)

print("\nShape:")
print(traffic.shape)

print("\nOriginal columns:")
print(traffic.columns.tolist())

print("\nData types:")
print(traffic.dtypes)

print("\nMissing values:")
print(traffic.isnull().sum())

display(traffic.head())

ORIGINAL DATASET INFORMATION

Shape:
(8936, 16)

Original columns:
['Date', 'Area Name', 'Road/Intersection Name', 'Traffic Volume', 'Average Speed', 'Travel Time Index', 'Congestion Level', 'Road Capacity Utilization', 'Incident Reports', 'Environmental Impact', 'Public Transport Usage', 'Traffic Signal Compliance', 'Parking Usage', 'Pedestrian and Cyclist Count', 'Weather Conditions', 'Roadwork and Construction Activity']

Data types:
Date                                   object
Area Name                              object
Road/Intersection Name                 object
Traffic Volume                          int64
Average Speed                         float64
Travel Time Index                     float64
Congestion Level                      float64
Road Capacity Utilization             float64
Incident Reports                        int64
Environmental Impact                  float64
Public Transport Usage                float64
Traffic Signal Compliance             float64
Parking

,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,70.632330,84.044600,85.403629,111,Clear,No
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,41.924899,91.407038,59.983689,100,Clear,No
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,44.662384,61.375541,95.466020,189,Clear,No
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,32.773123,75.547092,63.567452,111,Clear,No
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,35.092601,64.634762,93.155171,104,Clear,No


In [ ]:
# ==========================================
# STANDARDIZE COLUMN NAMES
# ==========================================

traffic.columns = (
    traffic.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

rename_map = {
    "traffic_volume": "vehicle_count",
    "road/intersection_name": "road_id",
    "road_intersection_name": "road_id",
    "intersection_name": "road_id",

    "avg_speed": "speed",
    "average_speed": "speed",

    "lat": "latitude",
    "lon": "longitude",
    "lng": "longitude"
}

traffic = traffic.rename(columns=rename_map)

print("Standardized columns:")
print(traffic.columns.tolist())

Standardized columns:
['date', 'area_name', 'road_id', 'vehicle_count', 'speed', 'travel_time_index', 'congestion_level', 'road_capacity_utilization', 'incident_reports', 'environmental_impact', 'public_transport_usage', 'traffic_signal_compliance', 'parking_usage', 'pedestrian_and_cyclist_count', 'weather_conditions', 'roadwork_and_construction_activity']


In [ ]:
# ==========================================
# REQUIRED DATASET FEATURES
# ==========================================

required_columns = [
    "area_name",
    "road_id",
    "vehicle_count",
    "speed"
]

missing_required = [
    col for col in required_columns
    if col not in traffic.columns
]

if missing_required:
    print("Missing required columns:")
    print(missing_required)

    print("\nAvailable columns:")
    print(traffic.columns.tolist())

    raise ValueError(
        "Required columns are missing. "
        "Check the rename_map in the previous cell."
    )

print("All essential traffic columns are available.")

All essential traffic columns are available.


In [ ]:
# ==========================================
# NUMERIC CONVERSION
# ==========================================

numeric_candidates = [
    "vehicle_count",
    "speed",
    "travel_time_index",
    "road_capacity_utilization",
    "incident_reports",
    "latitude",
    "longitude"
]

for col in numeric_candidates:

    if col in traffic.columns:

        traffic[col] = pd.to_numeric(
            traffic[col],
            errors="coerce"
        )

print("Numerical conversion completed.")

Numerical conversion completed.


In [ ]:
# ==========================================
# DATE / TIME PROCESSING
# ==========================================

if "date" in traffic.columns:

    traffic["date"] = pd.to_datetime(
        traffic["date"],
        errors="coerce"
    )

# --------------------------------------------------
# OPTION 1:
# If your dataset already has an hour column
# --------------------------------------------------

if "hour" in traffic.columns:

    traffic["hour"] = pd.to_numeric(
        traffic["hour"],
        errors="coerce"
    )

    print("Existing hour column used.")

# --------------------------------------------------
# OPTION 2:
# Extract hour only if date contains useful time data
# --------------------------------------------------

elif "date" in traffic.columns:

    extracted_hour = traffic["date"].dt.hour

    # If timestamps contain varying hours
    if extracted_hour.nunique(dropna=True) > 1:

        traffic["hour"] = extracted_hour

        print("Hour extracted from timestamp.")

    else:

        print(
            "Date column does not contain useful "
            "hour-of-day information."
        )

        print(
            "For prototype spatio-temporal demonstration, "
            "a synthetic hour variable will be created."
        )

        traffic["hour"] = (
            np.arange(len(traffic)) % 17
        ) + 6

        traffic["hour_is_synthetic"] = True

else:

    print(
        "No date/hour information found. "
        "Creating demonstration hours from 06:00 to 22:00."
    )

    traffic["hour"] = (
        np.arange(len(traffic)) % 17
    ) + 6

    traffic["hour_is_synthetic"] = True


traffic["hour"] = pd.to_numeric(
    traffic["hour"],
    errors="coerce"
)

print("\nHour range:")
print(
    traffic["hour"].min(),
    "to",
    traffic["hour"].max()
)

Date column does not contain useful hour-of-day information.
For prototype spatio-temporal demonstration, a synthetic hour variable will be created.

Hour range:
6 to 22


In [ ]:
# ==========================================
# CLEAN TEXT FEATURES
# ==========================================

traffic["area_name"] = (
    traffic["area_name"]
    .astype(str)
    .str.strip()
)

traffic["road_id"] = (
    traffic["road_id"]
    .astype(str)
    .str.strip()
)

print("Example areas:")
print(
    traffic["area_name"]
    .dropna()
    .unique()[:30]
)

Example areas:
['Indiranagar' 'Whitefield' 'Koramangala' 'M.G. Road' 'Jayanagar' 'Hebbal'
 'Yeshwanthpur' 'Electronic City']


In [ ]:
# ==========================================
# REPRESENTATIVE AREA COORDINATES
# ==========================================

AREA_COORDINATES = {

    "Majestic": (12.9767, 77.5713),
    "MG Road": (12.9756, 77.6066),
    "Indiranagar": (12.9784, 77.6408),
    "Koramangala": (12.9352, 77.6245),
    "Whitefield": (12.9698, 77.7500),
    "Electronic City": (12.8452, 77.6602),
    "Jayanagar": (12.9250, 77.5938),
    "BTM Layout": (12.9166, 77.6101),
    "Hebbal": (13.0358, 77.5970),
    "Yeshwanthpur": (13.0285, 77.5467),
    "Marathahalli": (12.9591, 77.6974),
    "Silk Board": (12.9177, 77.6233),
    "KR Puram": (13.0040, 77.6878),
    "Banashankari": (12.9255, 77.5468),
    "Rajajinagar": (12.9915, 77.5545),
    "Malleshwaram": (13.0035, 77.5647),
    "HSR Layout": (12.9116, 77.6389),
    "Bellandur": (12.9304, 77.6784),
    "Sarjapur": (12.8600, 77.7860),
    "JP Nagar": (12.9063, 77.5857),
    "Airport Road": (13.0550, 77.5940)
}
# ==========================================
# ADD LATITUDE / LONGITUDE IF REQUIRED
# ==========================================

if "latitude" not in traffic.columns:
    traffic["latitude"] = np.nan

if "longitude" not in traffic.columns:
    traffic["longitude"] = np.nan


def find_coordinates(area):

    area_clean = str(area).strip().lower()

    for name, coords in AREA_COORDINATES.items():

        if name.lower() == area_clean:
            return coords

    return (np.nan, np.nan)


missing_geo = (
    traffic["latitude"].isna()
    | traffic["longitude"].isna()
)

mapped_coordinates = traffic.loc[
    missing_geo,
    "area_name"
].apply(find_coordinates)


traffic.loc[
    missing_geo,
    "latitude"
] = mapped_coordinates.apply(
    lambda x: x[0]
)

traffic.loc[
    missing_geo,
    "longitude"
] = mapped_coordinates.apply(
    lambda x: x[1]
)


print(
    "Rows with valid coordinates:",
    traffic[
        ["latitude", "longitude"]
    ].dropna().shape[0]
)

print(
    "Rows without coordinates:",
    traffic[
        ["latitude", "longitude"]
    ].isna().any(axis=1).sum()
)

unmatched_areas = (
    traffic.loc[
        traffic["latitude"].isna()
        | traffic["longitude"].isna(),
        "area_name"
    ]
    .dropna()
    .unique()
)

print("Areas requiring coordinates:")
print(unmatched_areas)



Rows with valid coordinates: 7435
Rows without coordinates: 1501
Areas requiring coordinates:
['M.G. Road']


In [ ]:
# ==========================================
# ESSENTIAL CLEANING
# ==========================================

before = len(traffic)

traffic = traffic.dropna(
    subset=[
        "vehicle_count",
        "speed",
        "hour"
    ]
).copy()

after = len(traffic)

print("Rows before cleaning:", before)
print("Rows after cleaning :", after)
print("Rows removed        :", before - after)

Rows before cleaning: 8936
Rows after cleaning : 8936
Rows removed        : 0


In [ ]:
# ==========================================
# TRAFFIC DENSITY
# ==========================================

max_vehicle_count = max(
    traffic["vehicle_count"].max(),
    1
)

traffic["traffic_density"] = (
    traffic["vehicle_count"]
    / max_vehicle_count
).clip(0, 1)


# ==========================================
# SPEED PENALTY
# ==========================================

FREE_FLOW_SPEED = max(
    traffic["speed"].quantile(0.95),
    40
)

traffic["speed_penalty"] = (
    1
    - traffic["speed"] / FREE_FLOW_SPEED
).clip(0, 1)


# ==========================================
# TRAVEL TIME PENALTY
# ==========================================

if "travel_time_index" in traffic.columns:

    max_tti = max(
        traffic["travel_time_index"].max(),
        1
    )

    traffic["travel_penalty"] = (
        traffic["travel_time_index"]
        / max_tti
    ).clip(0, 1)

else:

    traffic["travel_penalty"] = 0


# ==========================================
# CAPACITY PENALTY
# ==========================================

if "road_capacity_utilization" in traffic.columns:

    capacity = (
        traffic["road_capacity_utilization"]
        .copy()
    )

    if capacity.max() > 1:
        capacity = capacity / 100

    traffic["capacity_penalty"] = (
        capacity.clip(0, 1)
    )

else:

    traffic["capacity_penalty"] = 0


# ==========================================
# INCIDENT PENALTY
# ==========================================

if "incident_reports" in traffic.columns:

    max_incidents = max(
        traffic["incident_reports"].max(),
        1
    )

    traffic["incident_penalty"] = (
        traffic["incident_reports"]
        / max_incidents
    ).clip(0, 1)

else:

    traffic["incident_penalty"] = 0


# ==========================================
# FINAL CONGESTION INDEX
# ==========================================

traffic["congestion"] = (

    0.30 * traffic["traffic_density"]

    + 0.30 * traffic["speed_penalty"]

    + 0.15 * traffic["travel_penalty"]

    + 0.20 * traffic["capacity_penalty"]

    + 0.05 * traffic["incident_penalty"]

).clip(0, 1)


print("Congestion index calculated.")

display(
    traffic[
        [
            "area_name",
            "vehicle_count",
            "speed",
            "congestion"
        ]
    ].head(10)
)

Congestion index calculated.


,area_name,vehicle_count,speed,congestion
0,Indiranagar,50590,50.230299,0.598939
1,Indiranagar,30825,29.377125,0.630290
2,Whitefield,7399,54.474398,0.223659
3,Koramangala,60874,43.817610,0.680181
4,Koramangala,57292,41.116763,0.689337
5,M.G. Road,47848,34.241963,0.685832
6,M.G. Road,36574,29.982430,0.661077
7,Jayanagar,25379,38.455179,0.565307
8,Jayanagar,25022,35.039373,0.576620
9,Hebbal,31760,56.904556,0.490745


In [ ]:
def congestion_level(value):

    if value < 0.25:
        return "Free Flow"

    elif value < 0.50:
        return "Moderate"

    elif value < 0.75:
        return "Congested"

    else:
        return "Severe"


traffic["congestion_status"] = (
    traffic["congestion"]
    .apply(congestion_level)
)

print(
    traffic[
        "congestion_status"
    ].value_counts()
)

congestion_status
Congested    5673
Moderate     2990
Severe        195
Free Flow      78
Name: count, dtype: int64


In [ ]:
audit_rows = []

for col in traffic.columns:

    audit_rows.append({

        "Feature": col,

        "Data Type":
            str(traffic[col].dtype),

        "Missing (%)":
            round(
                traffic[col]
                .isna()
                .mean() * 100,
                2
            ),

        "Unique Values":
            traffic[col].nunique(
                dropna=True
            )
    })


data_audit = pd.DataFrame(audit_rows)

display(data_audit)

,Feature,Data Type,Missing (%),Unique Values
0,date,datetime64[ns],0.0,952
1,area_name,object,0.0,8
2,road_id,object,0.0,16
3,vehicle_count,int64,0.0,8100
4,speed,float64,0.0,8572
5,travel_time_index,float64,0.0,3711
6,congestion_level,float64,0.0,5053
7,road_capacity_utilization,float64,0.0,2352
8,incident_reports,int64,0.0,11
9,environmental_impact,float64,0.0,8100


In [ ]:
print("=" * 60)
print("TRAFFIC DATASET SUMMARY")
print("=" * 60)

display(
    traffic[
        [
            "vehicle_count",
            "speed",
            "hour",
            "congestion"
        ]
    ].describe()
)

TRAFFIC DATASET SUMMARY


,vehicle_count,speed,hour,congestion
count,8936.000000,8936.000000,8936.000000,8936.000000
mean,29236.048120,39.447427,13.996307,0.546952
std,13001.808801,10.707244,4.898624,0.124213
min,4233.000000,20.000000,6.000000,0.176626
25%,19413.000000,31.775825,10.000000,0.453483
50%,27600.000000,39.199368,14.000000,0.569212
75%,38058.500000,46.644517,18.000000,0.644229
max,72039.000000,89.790843,22.000000,0.840236


In [ ]:
area_congestion = (
    traffic
    .groupby(
        "area_name",
        as_index=False
    )["congestion"]
    .mean()
    .sort_values(
        "congestion",
        ascending=False
    )
)

fig_area = px.bar(
    area_congestion,
    x="area_name",
    y="congestion",
    title="Area-wise Average Traffic Congestion",
    labels={
        "area_name": "City Area",
        "congestion":
            "Average Congestion Index"
    }
)

fig_area.update_layout(
    xaxis_tickangle=-45
)

fig_area.show()

In [ ]:
hourly_congestion = (
    traffic
    .groupby(
        "hour",
        as_index=False
    )["congestion"]
    .mean()
    .sort_values("hour")
)

fig_hour = px.line(
    hourly_congestion,
    x="hour",
    y="congestion",
    markers=True,
    title="Hourly Average Traffic Congestion",
    labels={
        "hour": "Hour of Day",
        "congestion":
            "Average Congestion Index"
    }
)

fig_hour.show()

In [ ]:
fig_scatter = px.scatter(
    traffic,
    x="vehicle_count",
    y="speed",
    color="congestion",
    hover_data=[
        "area_name",
        "road_id"
    ],
    title="Vehicle Count vs Average Speed",
    labels={
        "vehicle_count": "Vehicle Count",
        "speed": "Average Speed",
        "congestion":
            "Congestion Index"
    }
)

fig_scatter.show()

In [ ]:
fig_distribution = px.histogram(
    traffic,
    x="congestion",
    nbins=20,
    title="Distribution of Traffic Congestion Index",
    labels={
        "congestion":
            "Congestion Index"
    }
)

fig_distribution.show()

In [ ]:
# ==========================================
# DBSCAN DATA PREPARATION
# ==========================================

cluster_data = traffic.dropna(
    subset=[
        "latitude",
        "longitude",
        "hour",
        "congestion"
    ]
).copy()

if len(cluster_data) == 0:

    raise ValueError(
        "No rows contain valid latitude/"
        "longitude values. "
        "Check AREA_COORDINATES."
    )


cluster_features = cluster_data[
    [
        "latitude",
        "longitude",
        "hour",
        "congestion"
    ]
]


cluster_scaler = StandardScaler()

X_cluster = cluster_scaler.fit_transform(
    cluster_features
)


print(
    "Observations available for DBSCAN:",
    len(cluster_data)
)

Observations available for DBSCAN: 7435


In [ ]:
# ==========================================
# DBSCAN
# ==========================================

dbscan = DBSCAN(
    eps=0.65,
    min_samples=3
)

cluster_data["cluster"] = (
    dbscan.fit_predict(
        X_cluster
    )
)

print("DBSCAN completed.")

print(
    "\nCluster distribution:"
)

print(
    cluster_data[
        "cluster"
    ].value_counts().sort_index()
)

DBSCAN completed.

Cluster distribution:
cluster
-1       1
 0    1720
 1     942
 2    2537
 3     950
 4     734
 5     551
Name: count, dtype: int64


In [ ]:
# ==========================================
# DBSCAN RESULTS
# ==========================================

labels = (
    cluster_data["cluster"]
    .values
)

n_clusters = len(
    set(labels) - {-1}
)

n_noise = int(
    np.sum(labels == -1)
)

total_points = len(labels)

noise_percentage = (
    n_noise / total_points * 100
    if total_points > 0
    else 0
)


print("=" * 55)
print("DBSCAN CLUSTERING RESULTS")
print("=" * 55)

print(
    f"Number of clusters : "
    f"{n_clusters}"
)

print(
    f"Noise points       : "
    f"{n_noise}"
)

print(
    f"Total observations : "
    f"{total_points}"
)

print(
    f"Noise percentage   : "
    f"{noise_percentage:.2f}%"
)


# ==========================================
# SILHOUETTE SCORE
# ==========================================

mask = labels != -1

valid_X = X_cluster[mask]
valid_labels = labels[mask]


if (
    len(valid_labels) > 1
    and len(set(valid_labels)) >= 2
):

    silhouette = silhouette_score(
        valid_X,
        valid_labels
    )

    print(
        f"Silhouette Score   : "
        f"{silhouette:.4f}"
    )

else:

    silhouette = np.nan

    print(
        "Silhouette Score   : "
        "Not available because fewer "
        "than two valid clusters exist."
    )

DBSCAN CLUSTERING RESULTS
Number of clusters : 6
Noise points       : 1
Total observations : 7435
Noise percentage   : 0.01%
Silhouette Score   : 0.2188


In [ ]:
cluster_plot = px.scatter(
    cluster_data,
    x="longitude",
    y="latitude",
    color=(
        cluster_data["cluster"]
        .astype(str)
    ),
    hover_data=[
        "area_name",
        "road_id",
        "hour",
        "congestion"
    ],
    title=(
        "DBSCAN Spatio-Temporal "
        "Congestion Clusters"
    ),
    labels={
        "longitude": "Longitude",
        "latitude": "Latitude",
        "color": "Cluster"
    }
)

cluster_plot.show()

In [ ]:
# ==========================================
# RANDOM FOREST FEATURES
# ==========================================

model_features = [
    "hour",
    "vehicle_count",
    "speed"
]

optional_features = [
    "travel_time_index",
    "road_capacity_utilization",
    "incident_reports"
]

for feature in optional_features:

    if feature in traffic.columns:

        if traffic[feature].notna().sum() > 0:

            model_features.append(feature)


print(
    "Random Forest features:"
)

print(model_features)

Random Forest features:
['hour', 'vehicle_count', 'speed', 'travel_time_index', 'road_capacity_utilization', 'incident_reports']


In [ ]:
model_data = traffic.dropna(
    subset=(
        model_features
        + ["congestion"]
    )
).copy()


X = model_data[
    model_features
]

y = model_data[
    "congestion"
]


print(
    "Random Forest observations:",
    len(model_data)
)

display(X.head())

Random Forest observations: 8936


,hour,vehicle_count,speed,travel_time_index,road_capacity_utilization,incident_reports
0,6,50590,50.230299,1.500000,100.000000,0
1,7,30825,29.377125,1.500000,100.000000,1
2,8,7399,54.474398,1.039069,36.396525,0
3,9,60874,43.817610,1.500000,100.000000,1
4,10,57292,41.116763,1.500000,100.000000,3


In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

print(
    "Training observations:",
    len(X_train)
)

print(
    "Testing observations:",
    len(X_test)
)

Training observations: 7148
Testing observations: 1788


In [ ]:
traffic_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

traffic_model.fit(
    X_train,
    y_train
)

predictions = traffic_model.predict(
    X_test
)

print(
    "Random Forest training completed."
)

Random Forest training completed.


In [ ]:
# ==========================================
# RANDOM FOREST TEST RESULTS
# ==========================================

mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    predictions
)


print("=" * 55)
print("RANDOM FOREST — SINGLE SPLIT")
print("=" * 55)

print(f"MAE  : {mae:.6f}")
print(f"MSE  : {mse:.8f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

RANDOM FOREST — SINGLE SPLIT
MAE  : 0.003482
MSE  : 0.00002429
RMSE : 0.004929
R²   : 0.998411


In [ ]:
# ==========================================
# FIVE-FOLD CROSS VALIDATION
# ==========================================

rf_cv = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


r2_scores = cross_val_score(
    rf_cv,
    X,
    y,
    cv=kfold,
    scoring="r2"
)


mae_scores = -cross_val_score(
    rf_cv,
    X,
    y,
    cv=kfold,
    scoring="neg_mean_absolute_error"
)


print("=" * 55)
print("RANDOM FOREST — 5-FOLD CROSS VALIDATION")
print("=" * 55)


for i in range(5):

    print(
        f"Fold {i + 1}: "
        f"R² = {r2_scores[i]:.4f}, "
        f"MAE = {mae_scores[i]:.4f}"
    )


print()

print(
    f"Mean R²  : "
    f"{r2_scores.mean():.4f}"
)

print(
    f"R² SD    : "
    f"{r2_scores.std():.6f}"
)

print(
    f"Mean MAE : "
    f"{mae_scores.mean():.4f}"
)

print(
    f"MAE SD   : "
    f"{mae_scores.std():.6f}"
)

RANDOM FOREST — 5-FOLD CROSS VALIDATION
Fold 1: R² = 0.9984, MAE = 0.0035
Fold 2: R² = 0.9984, MAE = 0.0034
Fold 3: R² = 0.9985, MAE = 0.0033
Fold 4: R² = 0.9984, MAE = 0.0034
Fold 5: R² = 0.9985, MAE = 0.0034

Mean R²  : 0.9984
R² SD    : 0.000050
Mean MAE : 0.0034
MAE SD   : 0.000050


In [ ]:
# ==========================================
# TABLE 5.1
# ==========================================

single_split_results = pd.DataFrame({
    "Model": [
        "Random Forest Regression"
    ],
    "MAE": [mae],
    "MSE": [mse],
    "RMSE": [rmse],
    "R²": [r2]
})

print("TABLE 5.1 — SINGLE SPLIT RESULTS")
display(
    single_split_results.round(6)
)


# ==========================================
# TABLE 5.2
# ==========================================

cv_results = pd.DataFrame({
    "Fold": [
        "Fold 1",
        "Fold 2",
        "Fold 3",
        "Fold 4",
        "Fold 5"
    ],
    "R²": r2_scores,
    "MAE": mae_scores
})

print(
    "TABLE 5.2 — CROSS VALIDATION"
)

display(
    cv_results.round(6)
)


print(
    "Mean R²:",
    round(
        r2_scores.mean(),
        6
    )
)

print(
    "Mean MAE:",
    round(
        mae_scores.mean(),
        6
    )
)

TABLE 5.1 — SINGLE SPLIT RESULTS


,Model,MAE,MSE,RMSE,R²
0,Random Forest Regression,0.003482,0.000024,0.004929,0.998411


TABLE 5.2 — CROSS VALIDATION


,Fold,R²,MAE
0,Fold 1,0.998408,0.003485
1,Fold 2,0.998433,0.003413
2,Fold 3,0.998503,0.003343
3,Fold 4,0.998352,0.003368
4,Fold 5,0.998455,0.003437


Mean R²: 0.99843
Mean MAE: 0.003409


In [ ]:
prediction_df = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": predictions
})


fig_prediction = px.scatter(
    prediction_df,
    x="Actual",
    y="Predicted",
    title=(
        "Actual vs Predicted "
        "Traffic Congestion"
    ),
    labels={
        "Actual":
            "Actual Congestion",
        "Predicted":
            "Predicted Congestion"
    }
)


minimum = min(
    prediction_df["Actual"].min(),
    prediction_df["Predicted"].min()
)

maximum = max(
    prediction_df["Actual"].max(),
    prediction_df["Predicted"].max()
)


fig_prediction.add_shape(
    type="line",
    x0=minimum,
    y0=minimum,
    x1=maximum,
    y1=maximum,
    line=dict(
        dash="dash"
    )
)

fig_prediction.show()

In [ ]:
importance_df = pd.DataFrame({
    "Feature":
        X.columns,

    "Importance":
        traffic_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=True
)


fig_importance = px.bar(
    importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title=(
        "Random Forest "
        "Feature Importance"
    )
)

fig_importance.show()

display(
    importance_df.sort_values(
        "Importance",
        ascending=False
    )
)

,Feature,Importance
1,vehicle_count,0.502808
3,travel_time_index,0.294185
2,speed,0.147891
4,road_capacity_utilization,0.053125
5,incident_reports,0.001652
0,hour,0.000339


In [ ]:
# ==========================================
# AREA DEVELOPMENT ANALYSIS
# ==========================================

development = (
    traffic
    .groupby(
        "area_name",
        as_index=False
    )
    .agg(
        average_congestion=(
            "congestion",
            "mean"
        ),

        average_speed=(
            "speed",
            "mean"
        ),

        average_vehicle_count=(
            "vehicle_count",
            "mean"
        )
    )
)


def development_priority(c):

    if c >= 0.75:
        return "Critical"

    elif c >= 0.55:
        return "High"

    elif c >= 0.35:
        return "Medium"

    else:
        return "Low"


development[
    "development_priority"
] = (
    development[
        "average_congestion"
    ]
    .apply(
        development_priority
    )
)


development = development.sort_values(
    "average_congestion",
    ascending=False
)

display(development)

,area_name,average_congestion,average_speed,average_vehicle_count,development_priority
4,Koramangala,0.633428,36.062629,40832.253666,High
5,M.G. Road,0.598578,37.535189,35300.426382,High
2,Indiranagar,0.575079,38.644341,32284.388372,High
1,Hebbal,0.532846,40.149855,26533.232632,Medium
3,Jayanagar,0.522478,39.789692,24601.329923,Medium
6,Whitefield,0.480415,42.140866,21295.127389,Medium
7,Yeshwanthpur,0.451231,43.447576,18931.546322,Medium
0,Electronic City,0.422358,43.661789,16346.644928,Medium


In [ ]:
def recommend_development(
    congestion
):

    if congestion >= 0.75:

        return (
            "Critical priority: evaluate "
            "adaptive traffic signals, "
            "junction redesign, public "
            "transport priority, dedicated "
            "turn lanes and major capacity "
            "improvement feasibility."
        )

    elif congestion >= 0.55:

        return (
            "High priority: evaluate signal "
            "optimization, parking restrictions, "
            "turning lanes, bus priority and "
            "junction improvements."
        )

    elif congestion >= 0.35:

        return (
            "Medium priority: improve signal "
            "timing, parking management, "
            "traffic enforcement and public "
            "transport availability."
        )

    else:

        return (
            "Low priority: continue traffic "
            "monitoring and preventive "
            "maintenance."
        )


development[
    "recommendation"
] = (
    development[
        "average_congestion"
    ]
    .apply(
        recommend_development
    )
)

display(
    development[
        [
            "area_name",
            "average_congestion",
            "development_priority",
            "recommendation"
        ]
    ]
)

,area_name,average_congestion,development_priority,recommendation
4,Koramangala,0.633428,High,"High priority: evaluate signal optimization, p..."
5,M.G. Road,0.598578,High,"High priority: evaluate signal optimization, p..."
2,Indiranagar,0.575079,High,"High priority: evaluate signal optimization, p..."
1,Hebbal,0.532846,Medium,"Medium priority: improve signal timing, parkin..."
3,Jayanagar,0.522478,Medium,"Medium priority: improve signal timing, parkin..."
6,Whitefield,0.480415,Medium,"Medium priority: improve signal timing, parkin..."
7,Yeshwanthpur,0.451231,Medium,"Medium priority: improve signal timing, parkin..."
0,Electronic City,0.422358,Medium,"Medium priority: improve signal timing, parkin..."


In [ ]:
fig_development = px.bar(
    development,
    x="area_name",
    y="average_congestion",
    color="development_priority",
    title=(
        "Smart City Development "
        "Priority by Area"
    ),
    labels={
        "area_name":
            "City Area",

        "average_congestion":
            "Average Congestion",

        "development_priority":
            "Development Priority"
    },
    category_orders={
        "development_priority": [
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    }
)

fig_development.update_layout(
    xaxis_tickangle=-45
)

fig_development.show()

In [ ]:
fig_development = px.bar(
    development,
    x="area_name",
    y="average_congestion",
    color="development_priority",
    title=(
        "Smart City Development "
        "Priority by Area"
    ),
    labels={
        "area_name":
            "City Area",

        "average_congestion":
            "Average Congestion",

        "development_priority":
            "Development Priority"
    },
    category_orders={
        "development_priority": [
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    }
)

fig_development.update_layout(
    xaxis_tickangle=-45
)

fig_development.show()

In [ ]:
def traffic_color(
    congestion
):

    if congestion < 0.25:
        return "green"

    elif congestion < 0.50:
        return "orange"

    elif congestion < 0.75:
        return "red"

    else:
        return "darkred"

In [ ]:
# ==========================================
# PREPARE MAP DATA
# ==========================================

map_data = traffic.dropna(
    subset=[
        "latitude",
        "longitude",
        "congestion"
    ]
).copy()


if len(map_data) == 0:

    raise ValueError(
        "No geographic data available. "
        "Check latitude/longitude mapping."
    )


center_lat = (
    map_data["latitude"].mean()
)

center_lon = (
    map_data["longitude"].mean()
)


traffic_map = folium.Map(
    location=[
        center_lat,
        center_lon
    ],
    zoom_start=11
)


for _, row in map_data.iterrows():

    popup_text = f"""
    <b>Area:</b>
    {row['area_name']}<br>

    <b>Road:</b>
    {row['road_id']}<br>

    <b>Vehicle Count:</b>
    {row['vehicle_count']:.0f}<br>

    <b>Speed:</b>
    {row['speed']:.2f}<br>

    <b>Congestion:</b>
    {row['congestion']:.3f}<br>

    <b>Status:</b>
    {row['congestion_status']}
    """


    color = traffic_color(
        row["congestion"]
    )


    folium.CircleMarker(

        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=7,

        color=color,

        fill=True,

        fill_color=color,

        fill_opacity=0.75,

        popup=folium.Popup(
            popup_text,
            max_width=300
        )

    ).add_to(
        traffic_map
    )


traffic_map

In [ ]:
heat_map = folium.Map(
    location=[
        center_lat,
        center_lon
    ],
    zoom_start=11
)


heat_data = (
    map_data[
        [
            "latitude",
            "longitude",
            "congestion"
        ]
    ]
    .values
    .tolist()
)


HeatMap(
    heat_data,
    radius=20,
    blur=15,
    max_zoom=13
).add_to(
    heat_map
)


heat_map

In [ ]:
cluster_map = folium.Map(
    location=[
        cluster_data[
            "latitude"
        ].mean(),

        cluster_data[
            "longitude"
        ].mean()
    ],
    zoom_start=11
)


cluster_colors = [
    "blue",
    "purple",
    "green",
    "orange",
    "red",
    "cadetblue",
    "darkpurple",
    "darkgreen"
]


for _, row in cluster_data.iterrows():

    cluster = int(
        row["cluster"]
    )

    if cluster == -1:

        color = "gray"

    else:

        color = cluster_colors[
            cluster
            % len(cluster_colors)
        ]


    popup = (
        f"Area: {row['area_name']}<br>"
        f"Cluster: {cluster}<br>"
        f"Hour: {row['hour']}<br>"
        f"Congestion: "
        f"{row['congestion']:.3f}"
    )


    folium.CircleMarker(

        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=7,

        color=color,

        fill=True,

        fill_opacity=0.75,

        popup=popup

    ).add_to(
        cluster_map
    )


cluster_map

In [ ]:
# ==========================================
# SMART CITY SUMMARY
# ==========================================

city_average_congestion = (
    traffic[
        "congestion"
    ].mean()
)

city_average_speed = (
    traffic[
        "speed"
    ].mean()
)

city_vehicle_count = (
    traffic[
        "vehicle_count"
    ].mean()
)

critical_areas = (
    development[
        development[
            "development_priority"
        ] == "Critical"
    ]
)


print("=" * 60)
print("SMART CITY OPERATIONS CENTER")
print("=" * 60)

print(
    f"Average Congestion : "
    f"{city_average_congestion:.3f}"
)

print(
    f"Average Speed      : "
    f"{city_average_speed:.2f}"
)

print(
    f"Average Vehicles   : "
    f"{city_vehicle_count:.2f}"
)

print(
    f"DBSCAN Clusters    : "
    f"{n_clusters}"
)

print(
    f"DBSCAN Noise       : "
    f"{noise_percentage:.2f}%"
)

print(
    f"Random Forest R²   : "
    f"{r2:.4f}"
)

print(
    f"Random Forest MAE  : "
    f"{mae:.4f}"
)

print(
    f"Critical Areas     : "
    f"{len(critical_areas)}"
)

SMART CITY OPERATIONS CENTER
Average Congestion : 0.547
Average Speed      : 39.45
Average Vehicles   : 29236.05
DBSCAN Clusters    : 6
DBSCAN Noise       : 0.01%
Random Forest R²   : 0.9984
Random Forest MAE  : 0.0035
Critical Areas     : 0


In [ ]:
# ==========================================
# DASHBOARD FUNCTIONS
# ==========================================

def city_summary():

    return f"""
# Smart City Operations Center

### Traffic Intelligence

**Average Congestion:** {city_average_congestion:.3f}

**Average Vehicle Speed:** {city_average_speed:.2f}

**Average Vehicle Count:** {city_vehicle_count:.2f}

### Machine Learning

**Random Forest Test R²:** {r2:.4f}

**Random Forest Test MAE:** {mae:.4f}

**5-Fold Mean R²:** {r2_scores.mean():.4f}

### Spatio-Temporal Analysis

**DBSCAN Clusters:** {n_clusters}

**Noise Percentage:** {noise_percentage:.2f}%

**Silhouette Score:** {
    f"{silhouette:.4f}"
    if not np.isnan(silhouette)
    else "N/A"
}

### Development Planning

**Critical Priority Areas:** {len(critical_areas)}
"""


def area_analysis(area):

    row = development[
        development[
            "area_name"
        ] == area
    ]

    if row.empty:

        return (
            "No information available "
            "for this area."
        )

    row = row.iloc[0]

    return f"""
## {area}

**Average Congestion:** {row['average_congestion']:.3f}

**Average Speed:** {row['average_speed']:.2f}

**Average Vehicle Count:** {row['average_vehicle_count']:.2f}

**Development Priority:** {row['development_priority']}

### Recommendation

{row['recommendation']}
"""

In [ ]:
# ==========================================
# GRADIO DASHBOARD
# ==========================================

area_choices = sorted(
    development[
        "area_name"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


with gr.Blocks(
    title="Smart City SOC"
) as smart_city_app:

    gr.Markdown(
        """
        # AI-Driven Smart City Operations Center

        Urban Traffic Intelligence,
        Spatio-Temporal Clustering and
        Development Planning System
        """
    )


    with gr.Tab(
        "City Overview"
    ):

        summary_output = gr.Markdown(
            value=city_summary()
        )


    with gr.Tab(
        "Traffic Analytics"
    ):

        gr.Plot(
            value=fig_area
        )

        gr.Plot(
            value=fig_hour
        )


    with gr.Tab(
        "ML Analysis"
    ):

        gr.Plot(
            value=fig_prediction
        )

        gr.Plot(
            value=fig_importance
        )


    with gr.Tab(
        "Development Planner"
    ):

        area_dropdown = gr.Dropdown(
            choices=area_choices,
            value=(
                area_choices[0]
                if area_choices
                else None
            ),
            label="Select City Area"
        )

        analyze_button = gr.Button(
            "Analyze Area"
        )

        area_output = gr.Markdown()


        analyze_button.click(
            fn=area_analysis,
            inputs=area_dropdown,
            outputs=area_output
        )


    with gr.Tab(
        "Development Priorities"
    ):

        gr.Plot(
            value=fig_development
        )

        gr.Dataframe(
            value=development,
            interactive=False
        )


    with gr.Tab(
        "Traffic Dataset"
    ):

        gr.Dataframe(
            value=traffic,
            interactive=False
        )


    with gr.Tab(
        "System Architecture"
    ):

        gr.Markdown(
            """
### Smart City SOC Architecture

Traffic Dataset
↓
Data Cleaning
↓
Feature Engineering
↓
Congestion Index
↓
Spatio-Temporal DBSCAN
↓
Random Forest Regression
↓
Traffic Visualization
↓
Development Priority Engine
↓
Smart City Operations Center

### Future Integration

- Live traffic sensors
- CCTV vehicle detection
- Smart parking
- Air-quality monitoring
- Waste management
- Public transport
- Road safety
- Emergency response
- Flood monitoring
- Energy management
- Citizen complaints
- A*/Dijkstra route planning
            """
        )


smart_city_app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0d6d86463a724eb9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
